import sys
sys.path.append('/home/semik/projekty/noncanonical_introns/')

import pathlib
pathlib.Path().absolute()

import introns
import intron_comparison
from collections import defaultdict
from tabulate import tabulate
import numpy as np
import matplotlib.pyplot as plt
from Bio import motifs
from Bio.Seq import Seq
from math import sqrt
import logomaker
from itertools import groupby
import pickle
from Bio import SeqIO
import re
import math

In [2]:
import sys
sys.path.append('/home/semik/projekty/noncanonical_introns/')

import pathlib
pathlib.Path().absolute()

import introns
import intron_comparison
from collections import defaultdict
from tabulate import tabulate
import numpy as np
import matplotlib.pyplot as plt
from Bio import motifs
from Bio.Seq import Seq
from math import sqrt
import logomaker
from itertools import groupby
import pickle
from Bio import SeqIO
import re
import math

In [3]:
  
def stats(genome_eug,genes_eug):  
    conv_count=0
    nconv_count=0
    both_count=0
    all_count=0
    non_count=0
    conventional_classes=[0,0,0,0,0,0,0,0]
    nonconventional_classes=[0,0,0,0,0,0,0,0,0,0,0]

    for name, gene in list(genes_eug.items()):
        for intron in gene.introns:
            all_count+=1

            if intron.best_nonconv_var:
                nonconventional_classes[intron.best_nonconv_var-1]+=1
                nconv_count+=1

            if intron.best_conv_var:
                conventional_classes[intron.best_conv_var-1]+=1
                conv_count+=1
                if intron.best_nonconv_var:
                    both_count+=1
            elif not intron.best_nonconv_var:
                non_count+=1
    #print("\nKonwencjonalne: %d, niekonwencjonalne: %d, oba: %d inne: %d, wszystkie: %d" %(conv_count,nconv_count, both_count,non_count,all_count))
    return conventional_classes, nonconventional_classes, conv_count,nconv_count, both_count,non_count,all_count

def liczenie_ocen_genow(genes, genes_rev, conv_classes, nonconv_classes):
    ulamki_niekonw=[i/all_count for i in nonconv_classes]
    punktacja_niekonw = [1/sqrt(sum(ulamki_niekonw[:a+1])) for a in range(len(nonconv_classes))]
    
    kolejnosc=[j[0] for j in sorted(wagi.items(), key=lambda i: i[1])][1:]
    conv_classes=[conv_classes[k-1] for k in kolejnosc]
    ulamki_konw = [i/all_count for i in conv_classes]
    punktacja_konw = [1/sqrt(sum(ulamki_konw[:a+1])) for a in range(len(conv_classes))]
    #punktacja_konw = [1/sum(ulamki_konw[:a+1]) for a in range(len(conv_classes))]
    #punktacja_konw = [1/sum([sqrt(u) for u in ulamki_konw][:a+1]) for a in range(len(conv_classes))]
    wszystkiegeny=[]
    geny1, geny2= list(genes.items()), list(genes_rev.items())
    
    for i in range(len(geny1)):
        punkty_basic,punkty_rev = 0, 0
        ilosc_intronow_genu=len(geny1[i][1].introns)
        for j in range(ilosc_intronow_genu):
            
            int1,int2=geny1[i][1].introns[j], geny2[i][1].introns[j]
            oceny1, oceny2 = [], []
            if int1.best_conv_var: oceny1.append(punktacja_konw[int1.best_conv_var-1])
            if int2.best_conv_var: oceny2.append(punktacja_konw[int1.best_conv_var-1])
            if int1.best_nonconv_var: oceny1.append(punktacja_niekonw[int1.best_nonconv_var-1])
            if int2.best_nonconv_var: oceny2.append(punktacja_niekonw[int2.best_nonconv_var-1])
            
            punkty_basic = max(oceny1) if oceny1 else 0
            punkty_rev= max(oceny2) if oceny2 else 0
                
        if ilosc_intronow_genu: ocena=(punkty_basic-punkty_rev)/ilosc_intronow_genu
        else: ocena=0
        wszystkiegeny.append((geny1[i][1],geny2[i][1], ocena))
    return wszystkiegeny

def find_kgrams(string, k):
    def kmers(sequence, k):
        return [sequence[i: i + k] for i in range(len(sequence)-k+1)]
    kgrams = sorted(kmers(string, k))
    groups = [(k, len(list(g))) for k, g in groupby(kgrams)]
    return sorted(groups, key=lambda i: i[1], reverse=True)

In [8]:
genome_bugtest, genes_bugtest, genes_bugtest_rev = introns.HELP_load_default_genome_genes(species="bugtest", with_reversed=True)
# genome_EG, genes_EG, genes_EG_rev = introns.create_both(genom_EG, geny_EG, geny_EG_rev, "stringtie")
# genome_EH, genes_EH, genes_EH_rev = introns.create_both(genom_EH, geny_EH, geny_EH_rev, "stringtie")
genome_EL, genes_EL, genes_EL_rev = introns.HELP_load_default_genome_genes(species="EL", with_reversed=True)

Creating genes, sequences, introns, exons and introns' class prediction


100%|██████████| 7/7 [00:00<00:00, 713.20it/s]


[CREATE] Genes, sequences, introns, exons created, introns' classes predicted
Creating genes, sequences, introns, exons and introns' class prediction


100%|██████████| 7/7 [00:00<00:00, 1208.63it/s]

[CREATE] Genes, sequences, introns, exons created, introns' classes predicted
Creating genes, sequences, introns, exons and introns' class prediction



100%|██████████| 38192/38192 [01:02<00:00, 615.96it/s]


[CREATE] Genes, sequences, introns, exons created, introns' classes predicted
Creating genes, sequences, introns, exons and introns' class prediction


100%|██████████| 38192/38192 [01:02<00:00, 610.92it/s]

[CREATE] Genes, sequences, introns, exons created, introns' classes predicted


In [9]:
'''fasty do RNAfold'''
for gatunek, genome, genes in [("bugtest", genome_bugtest, genes_bugtest), ("E.longa",genome_EL, genes_EL)]:
    file="./%s_konwencjonalne.fasta" %gatunek
    with open(file, 'w') as f:
        for name,gene in genes.items():
            for intron in gene.introns:
                if intron.best_conv_var:
                    bco=intron.best_conv_obj
                    seq=bco.prev_exon.sequence[-5:]+bco.sequence[:20]+"AAAAAAAAAA"+bco.sequence[-20:]+bco.next_exon.sequence[:5]
                    f.write(">intron\n%s\n" %seq)


Ms=[]
classes=[]
for name,genome,genes,genes_rev in [("bugtest", genome_bugtest, genes_bugtest, genes_bugtest_rev), ("E. gracilis",genome_EG, genes_EG, genes_EG_rev), ("E. hiemalis", genome_EH, genes_EH, genes_EH_rev), ("E. longa",genome_EL, genes_EL, genes_EL_rev)]:
    conventional_classes, nonconventional_classes, conv_count,nconv_count, both_count,non_count,all_count = stats(genome, genes)
    conventional_classes_rev, nonconventional_classes_rev, conv_count_rev,nconv_count_rev, both_count_rev,non_count_rev,all_count_rev = stats(genome, genes_rev)
    M=[name, ["typ","konwencjonalne","niekonwencjonalne","oba","inne","wszystkie"],["dobre",conv_count,nconv_count, both_count,non_count,all_count],["reversed",conv_count_rev,nconv_count_rev, both_count_rev,non_count_rev,all_count_rev]]
    Ms.append(M)
    classes.append([[name, genes, genes_rev, all_count],[conventional_classes, nonconventional_classes], [conventional_classes_rev, nonconventional_classes_rev]])

wagi={0:0, 1:1, 2:3, 3:5, 4:7, 5:2, 6:4, 7:6, 8:8}

wszystkie_introny, wszystkie_introny_gat=[], []
introny_0, introny_0_gat=[],[]
introny_jednostronne, introny_jednostronne_gat=[],[]
for species in classes[1:]:
    nazwa=species[0][0]
    print("\n%s" %nazwa)
    all_count=species[0][3]
    conv_classes, nonconv_classes = species[1]
    
    geny_for, geny_rev = species[0][1], species[0][2]
    wszystkiegeny=liczenie_ocen_genow(geny_for, geny_rev, conv_classes, nonconv_classes)
    
    wszystkie_geny= [i[0] for i in wszystkiegeny]
    lista_wszystkich = [intron.best_nonconv_obj for i in [gen.introns for gen in wszystkie_geny] for intron in i if intron.best_nonconv_var]
    wszystkie_introny += lista_wszystkich
    wszystkie_introny_gat.append((nazwa,lista_wszystkich))
    geny_0 = [i[0] for i in wszystkiegeny if i[2]>0]
    lista_0 = [intron.best_nonconv_obj for i in [gen.introns for gen in geny_0] for intron in i if intron.best_nonconv_var]
    introny_0 += lista_0
    introny_0_gat.append((nazwa,lista_0))
    
    geny_jednostronne_for = [i[0] for i in wszystkiegeny if i[2]>0]
    geny_jednostronne_rev = [i[1] for i in wszystkiegeny if i[2]<0]
    lista_jednostronnych = [intron.best_nonconv_obj for i in [gen.introns for gen in geny_jednostronne_for] for intron in i if intron.best_nonconv_var] + [intron.best_nonconv_obj for i in [gen.introns for gen in geny_jednostronne_rev] for intron in i if intron.best_nonconv_var] 
    introny_jednostronne +=lista_jednostronnych
    introny_jednostronne_gat.append((nazwa, lista_jednostronnych))
wszystkie = [("all", wszystkie_introny), ("over0", introny_0), ('onesided', introny_jednostronne)]
gatunki = [("all", wszystkie_introny_gat), ("over0", introny_0_gat), ('onesided', introny_jednostronne_gat)]

In [10]:
'''Ms=[]
classes=[]
#for name,genome,genes,genes_rev in [("bugtest", genome_bugtest, genes_bugtest, genes_bugtest_rev), ("E. gracilis",genome_EG, genes_EG, genes_EG_rev), ("E. hiemalis", genome_EH, genes_EH, genes_EH_rev), ("E. longa",genome_EL, genes_EL, genes_EL_rev)]:
for name,genome,genes,genes_rev in [("E. gracilis", genome_EG, genes_EG, genes_EG_rev), ("E. hiemalis", genome_EH, genes_EH, genes_EH_rev), ("E. longa",genome_EL, genes_EL, genes_EL_rev)]:
    conventional_classes, nonconventional_classes, conv_count,nconv_count, both_count,non_count,all_count = stats(genome, genes)
    conventional_classes_rev, nonconventional_classes_rev, conv_count_rev,nconv_count_rev, both_count_rev,non_count_rev,all_count_rev = stats(genome, genes_rev)
    M=[name, ["typ","konwencjonalne","niekonwencjonalne","oba","inne","wszystkie"],["dobre",conv_count,nconv_count, both_count,non_count,all_count],["reversed",conv_count_rev,nconv_count_rev, both_count_rev,non_count_rev,all_count_rev]]
    Ms.append(M)
    classes.append([[name, genes, genes_rev, all_count],[conventional_classes, nonconventional_classes], [conventional_classes_rev, nonconventional_classes_rev]])

wagi={0:0, 1:1, 2:3, 3:5, 4:7, 5:2, 6:4, 7:6, 8:8}'''

all_file=open("/introny_fasty/all_introns.fasta", "w")
#over0_file=open("/introny_fasty/over0_introns.fasta", "w")
#onesided_file=open("/introny_fasty/onesided_introns.fasta", "w")

wszystkie_introny, wszystkie_introny_gat=[], []
introny_0, introny_0_gat=[],[]
introny_jednostronne, introny_jednostronne_gat=[],[]
for species in classes:
    nazwa=species[0][0]
    #print("\n%s" %nazwa)
    all_count=species[0][3]
    conv_classes, nonconv_classes = species[1]
    
    geny_for, geny_rev = species[0][1], species[0][2]
    wszystkiegeny=liczenie_ocen_genow(geny_for, geny_rev, conv_classes, nonconv_classes)
    for gen_for,gen_rev,ocena in wszystkiegeny[:20]:
        if ocena<0: continue
        strony=[None,None]
        for intron_for in gen_for.introns:
            if intron_for.best_conv_var:
                strony[0]=True
                break
        for intron_rev in gen_rev.introns:
            if intron_rev.best_conv_var:
                strony[1]=True
                break
        
        if strony==[True,None]:
            dobrygen=gen_for
                
        elif strony==[None,True]:
            dobrygen=gen_rev
    
        for intron in dobrygen.introns:
            if intron.best_conv_var and not intron.best_nonconv_var:
                konwencjonalnosc='KX'
                i=intron.best_conv_obj
            elif intron.best_nonconv_var and not intron.best_conv_var:
                konwencjonalnosc='XN'
                i=intron.best_nonconv_obj
            elif not intron.best_conv_var and not intron.best_nonconv_var:
                konwencjonalnosc='XX'
                i==intron
            else:
                konwencjonalnosc='KN'
                if introns.conventional_class_rate(intron.best_conv_var)<intron.best_nonconv_var:
                    #print(intron.best_conv_var, intron.best_nonconv_var)#, len(intron.best_conv_obj.sequence), len(intron.best_nonconv_obj.sequence))
                    i=intron.best_conv_obj
                else:
                    i=intron.best_nonconv_obj
            #onesided_file.write(">"+nazwagat.replace(" ", "")+"; "+str(i.scaffold_start)+" "+str(i.scaffold_end)+"; "+konwencjonalnosc+"\n"+i.sequence+"\n")
#onesided_file.close()
            
'''wszystkie_geny= [i[0] for i in wszystkiegeny]
    lista_wszystkich = [intron.best_nonconv_obj for i in [gen.introns for gen in wszystkie_geny] for intron in i if intron.best_nonconv_var]
    wszystkie_introny += lista_wszystkich
    wszystkie_introny_gat.append((nazwa,lista_wszystkich))
    geny_0 = [i[0] for i in wszystkiegeny if i[2]>0]
    geny_jednostronne_for = [i[0] for i in wszystkiegeny if i[2]>0]
    geny_jednostronne_rev = [i[1] for i in wszystkiegeny if i[2]<0]'''

FileNotFoundError: [Errno 2] No such file or directory: '/introny_fasty/all_introns.fasta'

Rysowanie loga

In [ ]:
'''loga z podzialem na gatunki'''

listaplikow = [("all", "./introny_fasty/gatunki_introny_all.fasta"), ("onesided","./introny_fasty/gatunki_introny_onesided.fasta"), ("over0", "./introny_fasty/gatunki_introny_over0.fasta"), ("unmoved-onesided","./introny_fasty/gatunki_introny_unmoved-onesided.fasta")]
for typlisty, plikfasta in listaplikow[-1:]:
    print(typlisty)
    poczatki, konce=[],[]
    for record in SeqIO.parse(plikfasta, "fasta"):
        seq_poczatek=str(record.seq[:25])
        seq_koniec = str(record.seq[-25:])
        if typlisty=="unmoved-onesided": nazwagat, q, klasa, klasaCon = record.description.split()
        else: nazwagat, q, klasa = record.description.split()
        poczatki.append((nazwagat, klasa, seq_poczatek))
        konce.append((nazwagat, klasa, seq_koniec))
    for gatunek in ['E.gracilis', 'E.hiemalis', 'E.longa']:
        inst1=[seq[2] for seq in poczatki if seq[0]==gatunek]
        inst2=[seq[2] for seq in konce if seq[0]==gatunek]
        
        with open('./weblogo/fastydolog/%s_%s_p.fasta' %(typlisty, gatunek), 'w') as file:
            for seq in inst1:
                file.write(">intron\n"+seq+"\n")
        file.close()
        
        with open('./weblogo/fastydolog/%s_%s_k.fasta' %(typlisty, gatunek), 'w') as file:
            for seq in inst2:
                file.write(">intron\n"+seq+"\n")
        file.close()
        '''
        def rysujlogo(inst, ax, gatunek, typlisty, koniec=False):
            instances=logomaker.transform_matrix(logomaker.alignment_to_matrix(inst), normalize_values=True)
            ss_logo = logomaker.Logo(instances, width=.9, vpad=.01, ax=ax, fade_probabilities=False, stack_order='small_on_top', color_scheme='classic')
            ss_logo.ax.set_title("Loga początkowych sekwencji intronów gatunku %s, %s" %(gatunek, typlisty))
            ss_logo.ax.set_ylabel('Prawdopodobieństwo')
            ss_logo.ax.set_label('Pozycja')
            pionowalinia=4.5 if koniec==False else 19.5
            ss_logo.ax.axvline(x=pionowalinia, color='k')
            ss_logo.ax.xaxis.set_ticks(range(0,25, 2))
            x_labels = [str(i) for i in range(-5,20, 2)] if koniec==False else [str(i) for i in range(-20,5,2)]
            ss_logo.ax.set_xticklabels(x_labels)
            
        fig,(ax1,ax2) = plt.subplots(nrows=1,ncols=2, figsize=(25,4))
        rysujlogo(inst1,ax1, gatunek,typlisty,koniec=False)
        rysujlogo(inst2,ax2, gatunek,typlisty,koniec=True)
        plt.show()'''

In [ ]:
'''loga z podzialem na klasy'''
listaplikow = [("all", "./introny_fasty/gatunki_introny_all.fasta")]#("all", "./introny_fasty/gatunki_introny_all.fasta"), ("onesided","./introny_fasty/gatunki_introny_onesided.fasta"), ("over0", "./introny_fasty/gatunki_introny_over0.fasta"), ("unmoved-onesided","./introny_fasty/gatunki_introny_unmoved-onesided.fasta")]
for typlisty, plikfasta in listaplikow:
    poczatki, konce=[],[]
    for record in SeqIO.parse(plikfasta, "fasta"):
        seq_poczatek=str(record.seq[:25])
        seq_koniec = str(record.seq[-25:])
        nazwagat, q, klasa = record.description.split()
        poczatki.append((nazwagat, klasa, seq_poczatek))
        konce.append((nazwagat, klasa, seq_koniec))
    for gatunek in ['E.gracilis', 'E.hiemalis', 'E.longa']:
        print(gatunek)
        inst21, inst22=[],[]
        for nrklasy in range(1,10):
            inst1=[seq[2] for seq in poczatki if seq[0:2]==(gatunek, str(nrklasy))]
            inst21+=inst1
            inst2=[seq[2] for seq in konce if seq[0:2]==(gatunek, str(nrklasy))]
            inst22+=inst2
            #print("klasy_%s_klasa%d_p" %(gatunek, nrklasy), "klasy_%s_klasa%d_k" %(gatunek, nrklasy))
            with open("./weblogo/fastydolog/klasy_%s_klasa%d_p.fasta" %(gatunek, nrklasy), 'w') as file:
                for seq in inst1:
                    file.write(">intron\n"+seq+"\n")
            file.close()
            with open("./weblogo/fastydolog/klasy_%s_klasa%d_k.fasta" %(gatunek, nrklasy), 'w') as file:
                for seq in inst2:
                    file.write(">intron\n"+seq+"\n")
            file.close()

        with open("./weblogo/fastydolog/klasy_%s_razem_p.fasta" %(gatunek), 'w') as file:
            for seq in inst21:
                file.write(">intron\n"+seq+"\n")
        file.close()
        with open("./weblogo/fastydolog/klasy_%s_klasy_k.fasta" %(gatunek), 'w') as file:
            for seq in inst22:
                file.write(">intron\n"+seq+"\n")
        file.close()
        '''
            def rysujlogo_klasy(inst, ax, gatunek, nrklasy, typlisty, pionowalinia=0):
                
                def Height(lista,n):
                    Hi=-sum([b*math.log(b,2) if b!=0 else 0 for b in lista])
                    return [b*math.log(4,2)-(Hi+(3/(2*n*math.log(n)))) for b in lista]

                nrklasy=str(nrklasy)
                instances=logomaker.transform_matrix(logomaker.alignment_to_matrix(inst))#, normalize_values=True)
                instances=instances.apply(lambda x: Height(x,len(poczatki)), axis=1, result_type='broadcast')
                ss_logo = logomaker.Logo(instances, width=.9, vpad=.01, ax=ax, fade_probabilities=False, stack_order='small_on_top', color_scheme='classic')
                ss_logo.ax.set_title("Loga początkowych sekwencji intronów gatunku %s, klasy %s, %s" %(gatunek, nrklasy, typlisty))
                ss_logo.ax.set_ylabel('Prawdopodobieństwo')
                ss_logo.ax.set_label('Pozycja')
                ss_logo.ax.axvline(x=pionowalinia, color='k')
                ss_logo.ax.xaxis.set_ticks(range(0,25, 2))
                
            fig,(ax1,ax2) = plt.subplots(nrows=1,ncols=2, figsize=(25,4))
            rysujlogo_klasy(inst1,ax1, gatunek, nrklasy, typlisty,4.5)
            rysujlogo_klasy(inst2,ax2, gatunek, nrklasy, typlisty, 19.5)
            plt.show()
        fig,(ax1,ax2) = plt.subplots(nrows=1,ncols=2, figsize=(25,4))
        rysujlogo_klasy(inst21,ax1, gatunek,"wszystkie", typlisty,4.5)
        rysujlogo_klasy(inst22,ax2, gatunek,"wszystkie", typlisty, 19.5)'''

In [ ]:
listaplikow = ["./introny_fasty/gatunki_introny_onesided.fasta", "./introny_fasty/gatunki_introny_over0.fasta", "./introny_fasty/gatunki_introny_all.fasta"] 

#tu robimy kmery
lista_slownikow=[]
for plikfasta in listaplikow:
    print(plikfasta)
    
    slownikkmerow={}
    for record in SeqIO.parse(plikfasta, "fasta"):
        seq=record.seq[5:-5]
        gat,typlisty,nrklasy=record.description.split(" ")
        for k in range(3,4): #3,19
                for pozycja in range(-20, -k):
                    if (nrklasy, gat, typlisty, k, pozycja) not in slownikkmerow.keys():
                        slownikkmerow[nrklasy, gat, typlisty, k, pozycja]=[seq[pozycja:pozycja+k]]
                    else:
                        slownikkmerow[nrklasy, gat, typlisty, k, pozycja].append(seq[pozycja:pozycja+k])
    def format_kmerdict_values(l):
        kg = [(k, len(list(g))) for k, g in groupby(sorted(l))]
        return sorted(kg, key=lambda i: i[1], reverse=True)


    ost_slownik = {x: format_kmerdict_values(v) for x,v in slownikkmerow.items()}
    lista_slownikow.append(ost_slownik)
    
    nazwapliku="kmery_"+re.split(r'/|\.', plikfasta)[3].replace('_introny', '')+".p"

In [ ]:
slownik_do_zapisania={}
for slownik in lista_slownikow:
    slownik_do_zapisania.update(slownik)
print(len(slownik_do_zapisania), sum([len(i) for i in lista_slownikow]))

with open('kmery_koniec.p', 'wb') as file:
     file.write(pickle.dumps(slownik_do_zapisania))
print("zapisane slownik_do_zapisania.p")

In [ ]:
#with open('./introny_fasty/kmery_poczatek.p', 'rb') as f:
with open('./introny_fasty/kmery_koniec.p', 'rb') as f:
    unpickler = pickle.Unpickler(f)
    #kp = unpickler.load()
    kk = unpickler.load()
#print(len(kp))
print(len(kk))